#Data Preprocessing - Text Classification

##Import Libraries

In [104]:
import random
import string
import re
import numpy as np
import pandas as pd

import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [105]:
REMOVE_NUMBERS = True      # True: Xóa số, False: Giữ nguyên
APPLY_STEMMING = False     # True: Dùng PorterStemmer, False: Không dùng

# Mức độ loại bỏ Stopword: 0.0 (0%), 0.2 (20%), 0.5 (50%), 0.8 (80%)
# Logic: Mức 0.0 sẽ không xóa gì.
# Các mức lớn hơn 0 sẽ sử dụng tập stopword kết hợp (NLTK + EDA)
# và dùng thông số này để lọc max_df trong TF-IDF (loại bỏ top % từ xuất hiện quá nhiều).
STOPWORD_LEVEL = 1.0

##Load Data

In [106]:
categories = ['comp.graphics', 'sci.med', 'rec.motorcycles', 'soc.religion.christian']

train_data = fetch_20newsgroups(
    subset='train',
    categories=categories,
    remove=('headers', 'footers', 'quotes')
)

test_data = fetch_20newsgroups(
    subset='test',
    categories=categories,
    remove=('headers', 'footers', 'quotes')
)

train_df = pd.DataFrame({
    'text': train_data.data,
    'target': train_data.target,
    'target_name': [train_data.target_names[i] for i in train_data.target]
})

test_df = pd.DataFrame({
    'text': train_data.data,
    'target': train_data.target,
    'target_name': [train_data.target_names[i] for i in train_data.target]
})

print(f"Train shape: {len(train_df)}")
print(f"Number of classes: {len(train_data.target_names)}")

Train shape: 2375
Number of classes: 4


##Filter EDA Outliers

In [107]:
train_df['word_count'] = train_df['text'].apply(lambda x: len(str(x).split()))
train_df['char_count'] = train_df['text'].apply(lambda x: len(str(x)))

train_df = train_df[train_df['word_count'] > 0]

In [108]:
train_df['punct_count'] = train_df['text'].apply(lambda x: len([c for c in x if c in string.punctuation]))
train_df['punct_ratio'] = train_df['punct_count'] / (train_df['char_count'] + 1)
train_df = train_df[train_df['punct_ratio'] < 0.5]

print(f"Number of documents after cleaning: {len(train_df)}")

Number of documents after cleaning: 2317


##Text Preprocessing Function

In [109]:
base_stopwords = set(stopwords.words('english'))
custom_stopwords = {'don', 've', 'like', 'just'}
ALL_STOPWORDS = base_stopwords.union(custom_stopwords)

stemmer = PorterStemmer()

def preprocess_text(text, drop_rate=0.8, remove_num=False, apply_stem=False):
    text = str(text).lower()
    if remove_num:
        text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'[^\w\s]', ' ', text)

    tokens = word_tokenize(text)

    # Find all stopwords in the current document
    stopword_indices = [i for i, word in enumerate(tokens) if word in ALL_STOPWORDS]

    # Calculate how many stopwords to drop
    num_to_drop = int(len(stopword_indices) * drop_rate)

    # Randomly select indices of stopwords to drop
    drop_indices = set(random.sample(stopword_indices, num_to_drop))

    # Keep non-stopwords and the un-dropped stopwords
    final_tokens = []
    for i, token in enumerate(tokens):
        if i not in drop_indices:
            final_tokens.append(stemmer.stem(token) if apply_stem else token)

    return ' '.join(final_tokens)

# Apply preprocessing to train set
train_df['cleaned_text'] = train_df['text'].apply(
    lambda x: preprocess_text(x, drop_rate=STOPWORD_LEVEL, remove_num=REMOVE_NUMBERS, apply_stem=APPLY_STEMMING)
)

In [110]:
train_df[['text', 'cleaned_text']].sample(5)

,text,cleaned_text
513,I have a few reprints left of chapters from my...,reprints left chapters book visions future inc...
619,"\nOne day out riding, my friend and i were pas...",one day riding friend passing field goats noti...
164,"\n\nConsiderably better than I feel about, say...",considerably better feel say punic wars pelopo...
1610,I've been thinking about the idea that was rai...,thinking idea raised michael covington think w...
1505,Anyone in Europe got any advice for a US citiz...,anyone europe got advice us citizen whose goin...


##TF-IDF Vectorization

In [111]:
# Fit and transform TF-IDF Vectorizer on train set
tfidf_vectorizer = TfidfVectorizer()

X_train_tfidf = tfidf_vectorizer.fit_transform(train_df['cleaned_text'])
y_train = train_df['target'].values

print(f"Train TF-IDF shape: {X_train_tfidf.shape}")

Train TF-IDF shape: (2317, 26749)


##Load and Process Test Data

In [112]:
test_df['cleaned_text'] = test_df['text'].apply(
    lambda x: preprocess_text(x, drop_rate=1.0, remove_num=REMOVE_NUMBERS, apply_stem=APPLY_STEMMING)
)

X_test_tfidf = tfidf_vectorizer.transform(test_df['cleaned_text'])
y_test = test_df['target'].values

print(f"Test TF-IDF shape: {X_test_tfidf.shape}")

Test TF-IDF shape: (2375, 26749)


## Model Testing (Optional)

In [113]:
# nb_model = MultinomialNB()
# nb_model.fit(X_train_tfidf, y_train)

# y_pred = nb_model.predict(X_test_tfidf)

# accuracy = accuracy_score(y_test, y_pred)
# print(f"Accuracy: {accuracy:.4f}\n")

# print("Classification Report:")
# print(classification_report(y_test, y_pred, target_names=categories))

Accuracy: 0.9655

Classification Report:
                        precision    recall  f1-score   support

         comp.graphics       0.98      0.96      0.97       584
               sci.med       0.99      0.96      0.98       598
       rec.motorcycles       1.00      0.95      0.97       594
soc.religion.christian       0.90      0.99      0.95       599

              accuracy                           0.97      2375
             macro avg       0.97      0.97      0.97      2375
          weighted avg       0.97      0.97      0.97      2375



## Summary

1. **Training Data (Train):**
* `X_train_tfidf`: The feature matrix of the training set.
* `y_train`: The actual labels (target) of the training set.


2. **Evaluation Data (Test):**
* `X_test_tfidf`: The feature matrix of the test set (transformed using the training set's configuration).
* `y_test`: The actual labels (target) of the test set.


3. **Supplementary Information:**
* `categories`: A list of 4 topic names (used to map numerical labels to text in the Confusion Matrix / Classification Report).
* `tfidf_vectorizer` & `preprocess_text`: Retained for preprocessing and transforming text entered by users later (Inference phase).